# Model Exploration

This notebook demonstrates how to use the ChatVLMLLM models for document OCR.

In [ ]:
# Add project root to path
import sys
sys.path.append('..')

from PIL import Image
from models import ModelLoader
from utils import ImageProcessor, TextExtractor
import matplotlib.pyplot as plt

## 1. Load Model

First, let's load a model. Start with GOT-OCR for basic OCR.

In [ ]:
# Load GOT-OCR model
print("Loading model...")
model = ModelLoader.load_model('got_ocr')
print(f"Model loaded: {model.get_model_info()}")

## 2. Load and Preprocess Image

In [ ]:
# Load image
image_path = '../examples/sample_document.jpg'  # Update with your image
image = Image.open(image_path)

# Display original
plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.axis('off')
plt.title('Original Image')
plt.show()

# Get image info
info = ImageProcessor.get_image_info(image)
print(f"Image info: {info}")

In [ ]:
# Preprocess image
processed_image = ImageProcessor.preprocess(
    image,
    resize=True,
    enhance=True,
    denoise=False
)

# Display processed
plt.figure(figsize=(12, 8))
plt.imshow(processed_image)
plt.axis('off')
plt.title('Processed Image')
plt.show()

## 3. Run OCR

In [ ]:
# Extract text
print("Running OCR...")
text = model.process_image(processed_image)

print("\n=== Extracted Text ===")
print(text)

## 4. Post-process Results

In [ ]:
# Clean text
cleaned_text = TextExtractor.clean_text(text)

print("\n=== Cleaned Text ===")
print(cleaned_text)

# Extract entities
dates = TextExtractor.extract_dates(cleaned_text)
emails = TextExtractor.extract_emails(cleaned_text)
phones = TextExtractor.extract_phone_numbers(cleaned_text)
amounts = TextExtractor.extract_amounts(cleaned_text)

print("\n=== Extracted Entities ===")
print(f"Dates: {dates}")
print(f"Emails: {emails}")
print(f"Phones: {phones}")
print(f"Amounts: {amounts}")

## 5. Field Extraction

In [ ]:
from utils.field_parser import FieldParser

# Parse as invoice (example)
fields = FieldParser.parse_invoice(cleaned_text)

print("\n=== Extracted Fields ===")
for field, value in fields.items():
    print(f"{field}: {value}")

## 6. Try Qwen2-VL for Intelligent Extraction

In [ ]:
# Load Qwen2-VL
qwen_model = ModelLoader.load_model('qwen_vl_2b')

# Ask specific question
prompt = """Extract the following information from this document:
- Document type
- Date
- Total amount
- Vendor/Issuer name

Provide the answer in JSON format."""

response = qwen_model.process_image(processed_image, prompt)

print("\n=== Qwen2-VL Response ===")
print(response)

## 7. Interactive Chat

In [ ]:
# Chat with the model
history = []

# First question
response1 = qwen_model.chat(processed_image, "What type of document is this?")
print(f"Q: What type of document is this?")
print(f"A: {response1}\n")
history.append({"role": "assistant", "content": response1})

# Follow-up question
response2 = qwen_model.chat(
    processed_image,
    "Can you extract all the monetary amounts?",
    history=history
)
print(f"Q: Can you extract all the monetary amounts?")
print(f"A: {response2}")

## 8. Performance Comparison

In [ ]:
import time

# Benchmark GOT-OCR
start = time.time()
got_result = model.process_image(processed_image)
got_time = time.time() - start

# Benchmark Qwen2-VL
start = time.time()
qwen_result = qwen_model.process_image(processed_image)
qwen_time = time.time() - start

print("\n=== Performance Comparison ===")
print(f"GOT-OCR: {got_time:.2f}s")
print(f"Qwen2-VL: {qwen_time:.2f}s")
print(f"Speed ratio: {qwen_time/got_time:.2f}x")

## 9. Cleanup

In [ ]:
# Unload models to free memory
ModelLoader.unload_all()
print("Models unloaded")